# Bobby Carrot — Per-Level PPO (Demo Run for L1–L10)

**Goal:** train one specialised PPO policy per normal level (1–10) so the agent can
demonstrate solving each level autonomously for the project review.

This notebook is **independent** of `train_colab.ipynb` (the phased L1–25 pipeline).
The phased pipeline stays intact for the post-review training target.

## What this notebook does

For every level 1–10 it calls `train_single_level(N)`, which runs PPO on just that
one map with tuned hyperparameters, early-stops at 95% rolling success, then
evaluates and backs the checkpoint up to Google Drive.

| Tier   | Levels  | Mechanics                           | Budget (cap) | lr    | entropy (start→floor) | ICM         |
|--------|---------|-------------------------------------|--------------|-------|-----------------------|-------------|
| **A1** | L1      | floor + carrot                      | 150k steps   | 3e-4  | 0.15 → 0.02           | off         |
| **A2** | L2–L3   | + crumble-intro (one-use bridges)   | 250k steps   | 2.5e-4| 0.12 → 0.06           | on (0.005)  |
| **B**  | L4–L7   | + crumble chains + first arrows     | 300k steps   | 2e-4  | 0.08 → 0.04           | on (0.01)   |
| **C**  | L8–L10  | + conveyor belts                    | 500k steps   | 2e-4  | 0.10 → 0.05           | on (0.02)   |

Early stopping typically cuts wall-clock in half on Tiers A1/A2.

### Why L2/L3 got their own tier (A2)

The earlier run classified L1–L3 as one tier and the resulting policy hit the
classic crumble-intro failure mode: `avg_reward ≈ 150` while `success = 0%`
(shaping gradient ≠ task gradient). The A2 preset extends the budget, raises
the entropy floor, enables ICM at a small scale, and lowers
`regression_trigger_drop` so the entropy boost re-arms on the first 20-point
drop rather than waiting for a 40-point collapse.

The environment's revisit/backtrack penalties were also softened — they were
punishing the optimal carrot-weave path on tight fields. See
`Game_Python/bobby_carrot/rl_env.py::RewardConfig` for the updated values.

## Anti-forgetting / curriculum

All disabled. One level → one policy → nothing to forget. See
`Bobby_Carrot/rl_models/single_level/level_configs.py` for the exact knob values.

---
## 1. Setup — Clone, install, mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO = '/content/Mini_Project_Game'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=False)
    print('Repo updated (git pull)')
else:
    subprocess.run(['git', 'clone', 'https://github.com/Charan20510/Mini_Project_Game.git', REPO], check=True)
    print('Repo cloned')
%cd /content/Mini_Project_Game

In [ ]:
%pip install torch numpy pandas matplotlib pillow --quiet

In [ ]:
import os, sys
os.chdir('/content/Mini_Project_Game')
for p in ('/content/Mini_Project_Game', '/content/Mini_Project_Game/Game_Python'):
    if p not in sys.path:
        sys.path.insert(0, p)
print('Working dir:', os.getcwd())

In [ ]:
# Drop stale .pyc so Colab always picks up the latest source after git pull.
import subprocess, importlib
subprocess.run(['find', '/content/Mini_Project_Game', '-name', '*.pyc', '-delete'], check=False)
subprocess.run(['find', '/content/Mini_Project_Game', '-name', '__pycache__', '-type', 'd', '-exec', 'rm', '-rf', '{}', '+'], check=False)
importlib.invalidate_caches()
print('pycache cleared')

In [ ]:
# Per-level Drive backup/restore. Each level has its own isolated directory.
DRIVE_ROOT = '/content/drive/MyDrive/Bobby_Carrot_RL/single_level'
os.makedirs(DRIVE_ROOT, exist_ok=True)

def restore_level(level_num: int) -> bool:
    """Copy checkpoints_level{N} from Drive if present. Returns True if restored."""
    import shutil
    src = f'{DRIVE_ROOT}/checkpoints_level{level_num}'
    dst = f'/content/Mini_Project_Game/checkpoints_level{level_num}'
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'✅ Restored checkpoints_level{level_num} from Drive')
        return True
    return False

def save_level(level_num: int) -> None:
    """Back up checkpoints_level{N} and logs_level{N} to Drive."""
    import shutil
    for kind in ('checkpoints', 'logs'):
        src = f'/content/Mini_Project_Game/{kind}_level{level_num}'
        dst = f'{DRIVE_ROOT}/{kind}_level{level_num}'
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f'💾 Saved {kind}_level{level_num} → Drive')

print('Drive helpers ready. DRIVE_ROOT =', DRIVE_ROOT)

In [ ]:
# Import the single-level training wrapper and the shared evaluator.
# Reload every module touched by training so a `git pull` inside the same
# kernel picks up the latest source — crucial when tuning reward shaping in
# rl_env.py between runs.
import importlib
import Bobby_Carrot.rl_models.config as _cfg
import Bobby_Carrot.rl_models.networks as _nets
import Bobby_Carrot.rl_models.ppo as _ppo
import Bobby_Carrot.rl_models.single_level.level_configs as _lc
import Bobby_Carrot.rl_models.single_level.train_single as _ts
import bobby_carrot.rl_env as _env
for m in (_env, _cfg, _nets, _ppo, _lc, _ts):
    importlib.reload(m)

from Bobby_Carrot.rl_models.single_level import train_single_level, best_ckpt_for_level, LEVEL_TIER
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

SUMMARY = {}  # level_num -> {'success_rate', 'avg_steps', 'checkpoint'}
print('Imports OK. LEVEL_TIER =', LEVEL_TIER)

In [ ]:
# One function, called from each per-level cell below. Keeps the level cells short.
def train_and_eval_level(level_num: int, episodes: int = 20, save_to_drive: bool = True):
    # Train (this blocks until early-stop or budget exhausted).
    train_single_level(level_num, checkpoint_root='.')

    # Evaluate the best checkpoint greedily over `episodes` runs.
    ckpt = best_ckpt_for_level(level_num, checkpoint_root='.')
    from Bobby_Carrot.rl_models.single_level.level_configs import build_configs_for_level
    _, train_cfg, _, _ = build_configs_for_level(level_num)
    print(f'\n\u2500\u2500\u2500 Evaluating normal-{level_num:02d} \u2500\u2500\u2500')
    r = evaluate_agent(
        algo='ppo', checkpoint_path=ckpt,
        levels=[('normal', level_num)],
        episodes_per_level=episodes,
        max_steps=train_cfg.max_steps_per_episode,
        observation_mode='full', device_str='auto',
    )
    agg = r['aggregate']
    SUMMARY[level_num] = {
        'success_rate': agg['success_rate'],
        'avg_steps': agg.get('avg_steps', 0),
        'checkpoint': ckpt,
    }
    print(f"\nLevel {level_num}: success={agg['success_rate']:.1%} | avg_steps={agg.get('avg_steps', 0):.1f}")
    if save_to_drive:
        save_level(level_num)
    return r

print('Helper ready: train_and_eval_level(N)')

---
## 2. Tier A — L1 (A1: pure carrot) and L2, L3 (A2: crumble-intro)

L1 early-stops under 50k steps on the old preset. L2 and L3 now use the A2
preset (250k cap, entropy floor 0.06, ICM on). Re-run a cell with
`train_single_level(N, total_timesteps_override=400_000)` if the log still
shows `success=0%` at the end of the default budget.

In [ ]:
train_and_eval_level(1)

In [ ]:
train_and_eval_level(2)

In [ ]:
train_and_eval_level(3)

---
## 3. Tier B — L4, L5, L6, L7 (crumble chains + first arrows)

Budget 300k steps each, ICM enabled.

In [ ]:
train_and_eval_level(4)

In [ ]:
train_and_eval_level(5)

In [ ]:
train_and_eval_level(6)

In [ ]:
train_and_eval_level(7)

---
## 4. Tier C — L8, L9, L10 (+ conveyor belts)

The hardest of the demo set. Budget 500k each, ICM on, rollout 4096. If any
level stalls below 95%, re-run its cell with `total_timesteps_override=750_000`.

```python
train_single_level(10, total_timesteps_override=750_000)
```

In [ ]:
train_and_eval_level(8)

In [ ]:
train_and_eval_level(9)

In [ ]:
train_and_eval_level(10)

---
## 5. Summary — success rates across all 10 levels

In [ ]:
print('=' * 60)
print(f"  {'Level':<6}{'Tier':<6}{'Success':<10}{'Avg steps':<12}")
print('-' * 60)
for lvl in range(1, 11):
    if lvl in SUMMARY:
        s = SUMMARY[lvl]
        print(f"  L{lvl:<5}{LEVEL_TIER[lvl]:<6}{s['success_rate']:<10.1%}{s['avg_steps']:<12.1f}")
    else:
        print(f"  L{lvl:<5}{LEVEL_TIER[lvl]:<6}{'NOT RUN':<10}{'-':<12}")
print('=' * 60)
if SUMMARY:
    avg_succ = sum(s['success_rate'] for s in SUMMARY.values()) / len(SUMMARY)
    print(f'\nAverage success across trained levels: {avg_succ:.1%}')

---
## 6. Demo playback

Colab has no display, so `render=True` will not open a window here. Two options:

### Option A — Save frame sequences on Colab, download, stitch into GIFs locally

The next cell runs each level once with `save_frames=True`, producing
`frames/normal-NN/*.png`. Download the folder and use any PNG-→-GIF tool (e.g.
`convert -delay 25 *.png demo.gif`).

### Option B — Download checkpoints + run evaluate locally for live rendering

After this notebook finishes, on your local machine:

```bash
# Copy checkpoints_level*/ from Drive to your local repo, then:
for L in 1 2 3 4 5 6 7 8 9 10; do
  python -m Bobby_Carrot.rl_models.evaluate \
    --algo ppo \
    --checkpoint checkpoints_level${L}/ppo/ppo_best.pt \
    --levels normal-${L} \
    --episodes 1 --max-steps 1500 \
    --render --render-fps 4
done
```

In [ ]:
# Option A: render-to-frames. Runs each trained level once, saves PNGs under ./frames/
from Bobby_Carrot.rl_models.single_level.level_configs import build_configs_for_level
import shutil

FRAMES_ROOT = '/content/Mini_Project_Game/frames'
os.makedirs(FRAMES_ROOT, exist_ok=True)

for lvl in sorted(SUMMARY.keys()):
    _, train_cfg, _, _ = build_configs_for_level(lvl)
    level_frames = f'{FRAMES_ROOT}/normal-{lvl:02d}'
    if os.path.isdir(level_frames):
        shutil.rmtree(level_frames)
    print(f'\n\u2500\u2500\u2500 Rendering L{lvl} frames \u2192 {level_frames} \u2500\u2500\u2500')
    evaluate_agent(
        algo='ppo', checkpoint_path=SUMMARY[lvl]['checkpoint'],
        levels=[('normal', lvl)], episodes_per_level=1,
        max_steps=train_cfg.max_steps_per_episode,
        observation_mode='full', device_str='auto',
        save_frames=True, frames_dir=level_frames,
    )

# Zip the whole frames/ folder for easy download.
shutil.make_archive('/content/frames_all', 'zip', FRAMES_ROOT)
print('\n\u2713 Frames zipped to /content/frames_all.zip — download via the Files pane.')

In [ ]:
# Final safety net: back up every level to Drive in case individual save_level calls were skipped.
for lvl in range(1, 11):
    save_level(lvl)

try:
    from google.colab import drive as _drive
    _drive.flush_and_unmount()
    print('\nDrive flushed.')
except Exception:
    pass